В цьому домашньому завданні ми знову працюємо з даними з нашого змагання ["Bank Customer Churn Prediction (DLU Course)"](https://www.kaggle.com/t/7c080c5d8ec64364a93cf4e8f880b6a0).

Тут ми побудуємо рішення задачі класифікації з використанням алгоритмів бустингу: XGBoost та LightGBM, а також використаємо бібліотеку HyperOpt для оптимізації гіперпараметрів.

0. Зчитайте дані `train.csv` в змінну `raw_df` та скористайтесь наведеним кодом нижче аби розділити дані на трнувальні та валідаційні і розділити дані на ознаки з матириці Х та цільову змінну. Назви змінних `train_inputs, train_targets, train_inputs, train_targets` можна змінити на ті, які Вам зручно.

  Наведений скрипт - частина отриманого мною скрипта для обробки даних. Ми тут не викнуємо масштабування та обробку категоріальних змінних, бо хочемо це делегувати алгоритмам, які будемо використовувати. Якщо щось не розумієте в наведених скриптах, рекомендую розібратись: навичка читати код - важлива складова роботи в машинному навчанні.

In [ ]:
import lightgbm as lgb
import numpy as np
import pandas as pd
import xgboost as xgb
from hyperopt import STATUS_OK, Trials, fmin, hp, tpe
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

from ml_homework.classification import compute_auroc
from ml_homework.paths import PROCESSED_DATA_DIR, RAW_DATA_DIR

In [2]:
def split_train_val(
    df: pd.DataFrame, target_col: str, test_size: float = 0.2, random_state: int = 42
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Split the dataframe into training and validation sets.

    Args:
        df (pd.DataFrame): The raw dataframe.
        target_col (str): The target column for stratification.
        test_size (float): The proportion of the dataset to include in the validation split.
        random_state (int): Random state for reproducibility.

    Returns:
        Tuple[pd.DataFrame, pd.DataFrame]: Training and validation dataframes.
    """
    train_df, val_df = train_test_split(
        df, test_size=test_size, random_state=random_state, stratify=df[target_col]
    )
    return train_df, val_df


def separate_inputs_targets(
    df: pd.DataFrame, input_cols: list, target_col: str
) -> tuple[pd.DataFrame, pd.Series]:
    """
    Separate inputs and targets from the dataframe.

    Args:
        df (pd.DataFrame): The dataframe.
        input_cols (list): List of input columns.
        target_col (str): Target column.

    Returns:
        Tuple[pd.DataFrame, pd.Series]: DataFrame of inputs and Series of targets.
    """
    inputs = df[input_cols].copy()
    targets = df[target_col].copy()
    return inputs, targets

In [ ]:
raw_df = pd.read_csv(RAW_DATA_DIR / "customer_churn/train.csv")

In [4]:
drop_cols = ["id", "CustomerId", "Surname"]

X = raw_df.drop(columns=["Exited", *drop_cols])
y = raw_df["Exited"]
input_cols = X.columns.to_list()
target_col = y.name

In [5]:
train_df, val_df = split_train_val(raw_df, target_col)

In [6]:
train_inputs, train_targets = separate_inputs_targets(train_df, input_cols, target_col)

val_inputs, val_targets = separate_inputs_targets(val_df, input_cols, target_col)

1. В тренувальному та валідаційному наборі перетворіть категоріальні ознаки на тип `category`. Можна це зробити двома способами:
 1. `df[col_name].astype('category')`, як було продемонстровано в лекції
 2. використовуючи метод `pd.Categorical(df[col_name])`

In [7]:
cat_features = train_inputs.select_dtypes(include=["object", "str"]).columns
train_inputs[cat_features] = train_inputs[cat_features].astype("category")
val_inputs[cat_features] = val_inputs[cat_features].astype("category")

In [8]:
train_inputs.dtypes

CreditScore         float64
Geography          category
Gender             category
Age                 float64
Tenure              float64
Balance             float64
NumOfProducts       float64
HasCrCard           float64
IsActiveMember      float64
EstimatedSalary     float64
dtype: object

2. Навчіть на отриманих даних модель `XGBoostClassifier`. Параметри алгоритму встановіть на свій розсуд, ми далі будемо їх тюнити. Рекомендую тренувати не дуже складну модель.

  Опис всіх конфігураційних параметрів XGBoostClassifier - тут https://xgboost.readthedocs.io/en/stable/parameter.html#global-config

  **Важливо:** зробіть такі налаштування `XGBoostClassifier` аби він самостійно обробляв незаповнені значення в даних і обробляв категоріальні колонки.

  Можна також, якщо працюєте в Google Colab, увімкнути можливість використання GPU (`Runtime -> Change runtime type -> T4 GPU`) і встановити параметр `device='cuda'` в `XGBoostClassifier` для пришвидшення тренування бустинг моделі.
  
  Після тренування моделі
  1. Виміряйте точність з допомогою AUROC на тренувальному та валідаційному наборах.
  2. Зробіть висновок про отриману модель: вона хороша/погана, чи є high bias/high variance?
  3. Порівняйте якість цієї моделі з тою, що ви отрмали з використанням DecisionTrees раніше. Чи вийшло покращити якість?

In [9]:
xgb_clf = XGBClassifier(
    max_depth=3,
    n_estimators=10,
    enable_categorical=True,
    missing=np.nan,
    device="cpu",
    random_state=42,
)

In [10]:
xgb_clf.fit(train_inputs, train_targets)
xgb_train_predictions = xgb_clf.predict(train_inputs)
xgb_val_predictions = xgb_clf.predict(val_inputs)

print(classification_report(train_targets, xgb_train_predictions, digits=4))
print(classification_report(val_targets, xgb_val_predictions, digits=4))

              precision    recall  f1-score   support

         0.0     0.9142    0.9646    0.9388      9558
         1.0     0.8235    0.6458    0.7239      2442

    accuracy                         0.8998     12000
   macro avg     0.8689    0.8052    0.8313     12000
weighted avg     0.8958    0.8998    0.8950     12000

              precision    recall  f1-score   support

         0.0     0.9142    0.9586    0.9359      2390
         1.0     0.7996    0.6475    0.7156       610

    accuracy                         0.8953      3000
   macro avg     0.8569    0.8031    0.8257      3000
weighted avg     0.8909    0.8953    0.8911      3000



In [11]:
train_xgb_auroc = compute_auroc(
    xgb_clf,
    train_inputs,
    train_targets,
    "train",
)
val_xgb_auroc = compute_auroc(
    xgb_clf,
    val_inputs,
    val_targets,
    "validation",
)

AUROC for train: 0.9330
AUROC for validation: 0.9318


XGBClassifier показала кращий результат, ніж дерево рішень: validation AUROC — 0.9318 проти 0.9172. Результати на тренувальних і валідаційних даних майже однакові, тому модель добре навчається без помітного перенавчання.

3. Використовуючи бібліотеку `Hyperopt` і приклад пошуку гіперпараметрів для `XGBoostClassifier` з лекції знайдіть оптимальні значення гіперпараметрів `XGBoostClassifier` для нашої задачі. Задайте свою сітку гіперпараметрів виходячи з тих параметрів, які ви б хотіли перебрати. Поставте кількість раундів в підборі гіперпараметрів рівну **20**.

  **Увага!** Для того, аби скористатись hyperopt, нам треба задати функцію `objective`. В ній ми маємо задати loss - це може будь-яка метрика, але бажано використовувтаи ту, яка цільова в вашій задачі. Чим менший лосс - тим ліпша модель на думку hyperopt. Тож, тут нам треба задати loss - негативне значення AUROC. В лекції ми натомість використовували Accuracy.

  Після успішного завершення пошуку оптимальних гіперпараметрів
    - виведіть найкращі значення гіперпараметрів
    - створіть в окремій зміній `final_clf` модель `XGBoostClassifier` з найкращими гіперпараметрами
    - навчіть модель `final_clf`
    - оцініть якість моделі `final_clf` на тренувальній і валідаційній вибірках з допомогою AUROC.
    - зробіть висновок про якість моделі. Чи стала вона краще порівняно з попереднім пунктом (2) цього завдання?

In [12]:
def xgb_objective(params):
    clf = xgb.XGBClassifier(
        n_estimators=int(params["n_estimators"]),
        learning_rate=params["learning_rate"],
        max_depth=int(params["max_depth"]),
        min_child_weight=params[
            "min_child_weight"
        ],  # Мінімальна сума ваг всіх вибірок, необхідна в кінцевому вузлі
        subsample=params[
            "subsample"
        ],  # Частка вибірок, що використовуються для побудови кожного дерева
        colsample_bytree=params[
            "colsample_bytree"
        ],  # Частка ознак, що використовуються при побудові кожного дерева
        gamma=params[
            "gamma"
        ],  # Мінімальне зменшення втрат, необхідне для виконання поділу
        reg_alpha=params["reg_alpha"],  # Параметр регуляризації L1 (Lasso)
        reg_lambda=params["reg_lambda"],  # Параметр регуляризації L2 (Ridge)
        enable_categorical=True,
        missing=np.nan,
        device="cpu",
        eval_metric="auc",
        early_stopping_rounds=10,
        random_state=42,
    )

    clf.fit(
        train_inputs,
        train_targets,
        eval_set=[(val_inputs, val_targets)],
        verbose=False,
    )
    val_auroc = compute_auroc(clf, val_inputs, val_targets, "validation")

    return {"loss": -val_auroc, "status": STATUS_OK}

In [13]:
# Простір гіперпараметрів
xgb_space = {
    "n_estimators": hp.quniform("n_estimators", 50, 500, 25),
    "learning_rate": hp.uniform("learning_rate", 0.01, 0.3),
    "max_depth": hp.quniform("max_depth", 3, 15, 1),
    "min_child_weight": hp.quniform("min_child_weight", 1, 10, 1),
    "subsample": hp.uniform("subsample", 0.5, 1.0),
    "colsample_bytree": hp.uniform("colsample_bytree", 0.5, 1.0),
    "gamma": hp.uniform("gamma", 0, 0.5),
    "reg_alpha": hp.uniform("reg_alpha", 0, 1),
    "reg_lambda": hp.uniform("reg_lambda", 0, 1),
}

In [14]:
# Оптимізація
xgb_trials = Trials()
best_xgb_params = fmin(
    fn=xgb_objective,
    space=xgb_space,
    algo=tpe.suggest,
    max_evals=20,
    trials=xgb_trials,
    rstate=np.random.default_rng(42),
)

# Перетворення значень гіперпараметрів у кінцеві типи
best_xgb_params["n_estimators"] = int(best_xgb_params["n_estimators"])
best_xgb_params["max_depth"] = int(best_xgb_params["max_depth"])
best_xgb_params["min_child_weight"] = int(best_xgb_params["min_child_weight"])

print("Найкращі гіперпараметри XGBoost:", best_xgb_params)

AUROC for validation: 0.9356                          
AUROC for validation: 0.9369                                                     
AUROC for validation: 0.9314                                                     
AUROC for validation: 0.9350                                                     
AUROC for validation: 0.9326                                                     
AUROC for validation: 0.9335                                                     
AUROC for validation: 0.9336                                                     
AUROC for validation: 0.9307                                                     
AUROC for validation: 0.9359                                                     
AUROC for validation: 0.9348                                                     
AUROC for validation: 0.9341                                                      
AUROC for validation: 0.9274                                                      
AUROC for validation: 0.9354             

In [15]:
# Навчання фінальної моделі з найкращими гіперпараметрами
final_xgb_clf = xgb.XGBClassifier(
    n_estimators=best_xgb_params["n_estimators"],
    learning_rate=best_xgb_params["learning_rate"],
    max_depth=best_xgb_params["max_depth"],
    min_child_weight=best_xgb_params["min_child_weight"],
    subsample=best_xgb_params["subsample"],
    colsample_bytree=best_xgb_params["colsample_bytree"],
    gamma=best_xgb_params["gamma"],
    reg_alpha=best_xgb_params["reg_alpha"],
    reg_lambda=best_xgb_params["reg_lambda"],
    enable_categorical=True,
    missing=np.nan,
    device="cpu",
    eval_metric="auc",
    early_stopping_rounds=10,
    random_state=42,
)
final_xgb_clf.fit(
    train_inputs,
    train_targets,
    eval_set=[(val_inputs, val_targets)],
    verbose=False,
)
final_xgb_auroc_train = compute_auroc(
    final_xgb_clf, train_inputs, train_targets, "train"
)
final_xgb_auroc_val = compute_auroc(
    final_xgb_clf, val_inputs, val_targets, "validation"
)

AUROC for train: 0.9448
AUROC for validation: 0.9374


Після підбору параметрів validation AUROC зріс з **0.9318** до **0.9374**. Результати на train (**0.9448**) і validation близькі, тому помітного перенавчання немає, а оптимізована модель стала трохи кращою.

4. Навчіть на наших даних модель LightGBM. Параметри алгоритму встановіть на свій розсуд, ми далі будемо їх тюнити. Рекомендую тренувати не дуже складну модель.

  Опис всіх конфігураційних параметрів LightGBM - тут https://lightgbm.readthedocs.io/en/latest/Parameters.html

  **Важливо:** зробіть такі налаштування LightGBM аби він самостійно обробляв незаповнені значення в даних і обробляв категоріальні колонки.

  Аби передати категоріальні колонки в LightGBM - необхідно виявити їх індекси і передати в параметрі `cat_feature=cat_feature_indexes`

  Після тренування моделі
  1. Виміряйте точність з допомогою AUROC на тренувальному та валідаційному наборах.
  2. Зробіть висновок про отриману модель: вона хороша/погана, чи є high bias/high variance?
  3. Порівняйте якість цієї моделі з тою, що ви отрмали з використанням XGBoostClassifier раніше. Чи вийшло покращити якість?

In [16]:
cat_feature_indexes = [train_inputs.columns.get_loc(column) for column in cat_features]

In [17]:
cat_feature_indexes

[1, 2]

In [18]:
lgb_clf = lgb.LGBMClassifier(
    max_depth=3,
    n_estimators=50,
    learning_rate=0.1,
    random_state=42,
    verbosity=-1,
)

lgb_clf.fit(
    train_inputs,
    train_targets,
    categorical_feature=cat_feature_indexes,
)

lgb_train_predictions = lgb_clf.predict(train_inputs)
lgb_val_predictions = lgb_clf.predict(val_inputs)

print(classification_report(train_targets, lgb_train_predictions, digits=4))
print(classification_report(val_targets, lgb_val_predictions, digits=4))

lgb_train_auroc = compute_auroc(lgb_clf, train_inputs, train_targets, "train")
lgb_val_auroc = compute_auroc(lgb_clf, val_inputs, val_targets, "validation")

              precision    recall  f1-score   support

         0.0     0.9184    0.9632    0.9403      9558
         1.0     0.8219    0.6650    0.7352      2442

    accuracy                         0.9025     12000
   macro avg     0.8701    0.8141    0.8377     12000
weighted avg     0.8988    0.9025    0.8985     12000

              precision    recall  f1-score   support

         0.0     0.9175    0.9590    0.9378      2390
         1.0     0.8048    0.6623    0.7266       610

    accuracy                         0.8987      3000
   macro avg     0.8612    0.8106    0.8322      3000
weighted avg     0.8946    0.8987    0.8949      3000

AUROC for train: 0.9388
AUROC for validation: 0.9362


Базова модель LightGBM показала гарний результат: validation AUROC дорівнює **0.9362**. Результати на train (**0.9388**) і validation майже однакові, тому помітного перенавчання немає. Якість трохи нижча за оптимізований XGBoost (**0.9374**).

5. Використовуючи бібліотеку `Hyperopt` знайдіть оптимальні значення гіперпараметрів `LightGBM`. Проведіть **10** раундів пошуку, виведіть найкращі параметри, навчіть `final_lgb_clf`, оцініть AUROC на train і validation та порівняйте результат із базовою моделлю із завдання 4.

In [19]:
def lgb_objective(params):
    clf = lgb.LGBMClassifier(
        n_estimators=int(
            params["n_estimators"]
        ),  # Кількість дерев у ансамблі (кількість ітерацій бустингу)
        learning_rate=params[
            "learning_rate"
        ],  # Коефіцієнт, на який зменшується внесок кожного доданого дерева
        max_depth=int(params["max_depth"]),  # Максимальна глибина кожного дерева
        num_leaves=int(
            params["num_leaves"]
        ),  # Максимальна кількість листків, що дозволяємо кожному дереву мати.
        min_child_weight=params[
            "min_child_weight"
        ],  # Мінімальна сума ваг всіх вибірок, необхідна в кінцевому вузлі
        subsample=params[
            "subsample"
        ],  # Частка вибірок, що використовуються для побудови кожного дерева
        colsample_bytree=params[
            "colsample_bytree"
        ],  # Частка ознак, що використовуються при побудові кожного дерева
        reg_alpha=params["reg_alpha"],  # Параметр регуляризації L1 (Lasso)
        reg_lambda=params["reg_lambda"],  # Параметр регуляризації L2 (Ridge)
        min_split_gain=params[
            "min_split_gain"
        ],  # Мінімальне зменшення втрат, необхідне для виконання поділу
        random_state=42,
        verbosity=-1,
    )

    clf.fit(
        train_inputs,
        train_targets,
        categorical_feature=cat_feature_indexes,
    )
    val_auroc = compute_auroc(clf, val_inputs, val_targets, "validation")

    return {"loss": -val_auroc, "status": STATUS_OK}

In [20]:
# Простір гіперпараметрів
lgb_space = {
    "n_estimators": hp.quniform("n_estimators", 50, 500, 25),
    "learning_rate": hp.uniform("learning_rate", 0.01, 0.3),
    "max_depth": hp.quniform("max_depth", 3, 15, 1),
    "num_leaves": hp.quniform("num_leaves", 20, 150, 1),
    "min_child_weight": hp.quniform("min_child_weight", 1, 10, 1),
    "subsample": hp.uniform("subsample", 0.5, 1.0),
    "colsample_bytree": hp.uniform("colsample_bytree", 0.5, 1.0),
    "reg_alpha": hp.uniform("reg_alpha", 0, 1),
    "reg_lambda": hp.uniform("reg_lambda", 0, 1),
    "min_split_gain": hp.uniform(
        "min_split_gain", 0, 0.1
    ),  # додано мінімальне зменшення втрат для поділу
}

In [21]:
# Оптимізація
lgb_trials = Trials()
best_lgb_params = fmin(
    fn=lgb_objective,
    space=lgb_space,
    algo=tpe.suggest,
    max_evals=10,
    trials=lgb_trials,
    rstate=np.random.default_rng(42),
)

# Перетворення значень гіперпараметрів у кінцеві типи
best_lgb_params["n_estimators"] = int(best_lgb_params["n_estimators"])
best_lgb_params["max_depth"] = int(best_lgb_params["max_depth"])
best_lgb_params["num_leaves"] = int(best_lgb_params["num_leaves"])
best_lgb_params["min_child_weight"] = int(best_lgb_params["min_child_weight"])

print("Найкращі гіперпараметри LightGBM:", best_lgb_params)

AUROC for validation: 0.9222                          
AUROC for validation: 0.9358                                                     
AUROC for validation: 0.9287                                                     
AUROC for validation: 0.9324                                                     
AUROC for validation: 0.9212                                                     
AUROC for validation: 0.9227                                                     
AUROC for validation: 0.9294                                                     
AUROC for validation: 0.9290                                                     
AUROC for validation: 0.9176                                                     
AUROC for validation: 0.9287                                                     
100%|██████████| 10/10 [00:13<00:00,  1.40s/trial, best loss: -0.9357853762260786]
Найкращі гіперпараметри LightGBM: {'colsample_bytree': np.float64(0.6043143849848778), 'learning_rate': np.float64(0.1091430

In [22]:
# Навчання фінальної моделі з найкращими гіперпараметрами
final_lgb_clf = lgb.LGBMClassifier(
    n_estimators=best_lgb_params["n_estimators"],
    learning_rate=best_lgb_params["learning_rate"],
    max_depth=best_lgb_params["max_depth"],
    num_leaves=best_lgb_params["num_leaves"],
    min_child_weight=best_lgb_params["min_child_weight"],
    subsample=best_lgb_params["subsample"],
    colsample_bytree=best_lgb_params["colsample_bytree"],
    reg_alpha=best_lgb_params["reg_alpha"],
    reg_lambda=best_lgb_params["reg_lambda"],
    min_split_gain=best_lgb_params["min_split_gain"],
    random_state=42,
    verbosity=-1,
)

final_lgb_clf.fit(
    train_inputs,
    train_targets,
    categorical_feature=cat_feature_indexes,
)
final_lgb_auroc_train = compute_auroc(
    final_lgb_clf, train_inputs, train_targets, "train"
)
final_lgb_auroc_val = compute_auroc(
    final_lgb_clf, val_inputs, val_targets, "validation"
)

AUROC for train: 0.9514
AUROC for validation: 0.9358


### Висновок після тюнінгу LightGBM

Після тюнінгу validation AUROC становить **0.9358**, що трохи нижче за результат базового LightGBM (**0.9362**). Отже, перевірені параметри не покращили модель, тому базовий варіант поки є кращим.

6. Оберіть модель з експериментів в цьому ДЗ і зробіть новий `submission` на Kaggle та додайте код для цього і скріншот скора на публічному лідерборді.
  
  **Напишіть коментар, чому ви обрали саме цю модель?**

  І я вас вітаю - це останнє завдання з цим набором даних 💪 На цьому етапі корисно проаналізувати, які моделі показали себе найкраще і подумати, чому.

In [25]:
test_df = pd.read_csv(RAW_DATA_DIR / "customer_churn/test.csv")
test_inputs = test_df[input_cols].copy()

test_inputs[cat_features] = test_inputs[cat_features].astype("category")

test_probabilities = final_xgb_clf.predict_proba(test_inputs)[:, 1]

submission = pd.DataFrame(
    {
        "id": test_df["id"],
        target_col: test_probabilities,
    }
)

submission_path = PROCESSED_DATA_DIR / "xgb_tuned_submission.csv"
submission.to_csv(submission_path, index=False)

submission.head()

,id,Exited
0,15000,0.120990
1,15001,0.035641
2,15002,0.105923
3,15003,0.450291
4,15004,0.054267


Для submission обрано оптимізований XGBoost, оскільки він показав найкращий validation AUROC — 0.9374. Результати на train і validation близькі, тому модель не має помітного перенавчання.